In [ ]:
#@title Libraries
import pandas as pd
import duckdb
import numpy as np
import tabulate
import time
import requests
from google.colab import drive, userdata

from openai import OpenAI

import hashlib
from datetime import datetime
#!pip install chromadb
import chromadb

import json
import os
from datetime import datetime, timezone

#Save duckdb to google drive for access
drive.mount('/content/drive')
con = duckdb.connect("/content/drive/MyDrive/nfl_rag.duckdb")

#LLM Connection
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

#Tables getting truncated is annoying
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)



In [ ]:
# @title Populate SQL database

# #play by play
# seasons = range(2015, 2026)

# for season in seasons:
#     url = f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{season}.parquet"
#     r = requests.get(url)
#     with open(f"pbp_{season}.parquet", "wb") as f:
#         f.write(r.content)

# con.execute("""
# CREATE TABLE play_by_play AS
# SELECT * FROM read_parquet('pbp_*.parquet');
# """)


# #players

# url = f"https://github.com/nflverse/nflverse-data/releases/download/players/players.parquet"
# r = requests.get(url)
# with open(f"players.parquet", "wb") as f:
#     f.write(r.content)

# con.execute("""
# CREATE TABLE players AS
# SELECT * FROM read_parquet('players.parquet');
# """)


# # weekly player stats

# seasons = range(2015, 2025)
# for season in seasons:
#     url = f"https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_{season}.parquet"
#     r = requests.get(url)

#     if r.status_code != 200:
#         print(f"Failed for {season}")
#         continue

#     with open(f"player_stats_{season}.parquet", "wb") as f:
#         f.write(r.content)

# con.execute("""
# CREATE TABLE player_stats AS
# SELECT * FROM read_parquet('player_stats_*.parquet');
# """)

# # snap counts
# for season in seasons:
#     url = f"https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_{season}.parquet"
#     r = requests.get(url)
#     with open(f"snap_counts_{season}.parquet", "wb") as f:
#         f.write(r.content)

# con.execute("""
# CREATE TABLE snap_counts AS
# SELECT * FROM read_parquet('snap_counts_*.parquet');
# """)




In [ ]:
#@title Create News RAG chromadb
CHROMA_PATH = "/content/drive/MyDrive/nfl_chroma"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
news_collection = chroma_client.get_or_create_collection(
    name="nfl_news",
    metadata={"hnsw:space": "cosine"}
)

def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

In [ ]:
# @title News — ESPN article fetcher

ESPN_NEWS_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/news"

def fetch_espn_news(limit: int = 100) -> list[dict]:
    """
    Fetch recent NFL news articles from ESPN's unofficial API.
    No API key required.

    Returns a list of dicts with keys:
      headline, description, published, categories (player names mentioned), url
    """
    all_articles = []
    page = 1
    per_page = 25  # ESPN's max per request

    while len(all_articles) < limit:
        params = {"limit": per_page, "page": page}
        try:
            r = requests.get(ESPN_NEWS_URL, params=params, timeout=10)
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"ESPN fetch failed on page {page}: {e}")
            break

        data     = r.json()
        articles = data.get("articles", [])
        if not articles:
            break  # no more articles

        for article in articles:
            # Extract player/team categories ESPN tags on each article
            categories = article.get("categories", [])
            players_mentioned = [
                cat.get("athlete", {}).get("displayName", "")
                for cat in categories
                if cat.get("type") == "athlete" and cat.get("athlete")
            ]
            teams_mentioned = [
                cat.get("team", {}).get("abbreviation", "")
                for cat in categories
                if cat.get("type") == "team" and cat.get("team")
            ]

            all_articles.append({
                "headline":         article.get("headline", ""),
                "description":      article.get("description", ""),
                "published":        article.get("published", ""),
                "players_mentioned": players_mentioned,
                "teams_mentioned":   teams_mentioned,
                "url":              article.get("links", {}).get("web", {}).get("href", ""),
            })

        if len(articles) < per_page:
            break  # ESPN returned fewer than requested — we've hit the end
        page += 1

    print(f"[ESPN] Fetched {len(all_articles)} articles.")
    return all_articles[:limit]


print("ESPN fetcher defined.")

# Part 1 — Stats Agent

## Design

The stats agent answers: *"How good/bad is this player historically?"*

It works from the `player_stats` table

The agent gets two tools:
- `get_schemas` — To inform SQL queries of database structure
- `run_query` — query the sql database with guardrails to ensure select statements only


In [29]:
#@title Get table schema and define query guardrails

ALLOWED_TABLES = {
    "player_stats",
    "snap_counts"
}
DB_PATH = "/content/drive/MyDrive/nfl_rag.duckdb"


def get_schemas(table_names: list[str]) -> str:
    invalid = [t for t in table_names if t not in ALLOWED_TABLES]
    if invalid:
        raise ValueError(f"Tables not allowed: {invalid}")

    cat_columns = {"season_type", "position", "position_group"}

    con = duckdb.connect(DB_PATH)
    try:
        output = []
        for table in table_names:
            df = con.execute(f"DESCRIBE {table}").df()
            schema_md = df.to_markdown(index=False)
            output.append(f"### {table}\n{schema_md}\n")

            # Append distinct values for known categorical columns
            existing_cats = cat_columns & set(df["column_name"].tolist())
            for col in existing_cats:
                vals = con.execute(
                    f"SELECT DISTINCT {col} FROM {table} WHERE {col} IS NOT NULL LIMIT 10"
                ).fetchall()
                vals_str = ", ".join(str(r[0]) for r in vals)
                output.append(f"  >> {col} values: {vals_str}\n")

    finally:
        con.close()

    return "\n".join(output)


def run_query(sql: str):
    # Basic guardrails
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE"]
    if any(word in sql.upper() for word in forbidden):
        raise ValueError("Dangerous SQL detected.")

    con = duckdb.connect(DB_PATH)
    try:
        result = con.execute(sql).df()
    finally:
        con.close()

    return result


In [30]:
#@title Define Stats Agent Prompt and tools
stat_tools = [
    {
        "type": "function",
        "function": {
            "name": "run_query",
            "description": "Run a SQL query against an allowed table.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "A SELECT query."
                    }
                },
                "required": ["sql"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "get_schemas",
        "description": (
            "Retrieve schemas (column names and types) for one or more tables. "
            "Allowed tables: player_stats, snap_counts."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "table_names": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": [
                            "player_stats",
                            "snap_counts"
                        ]
                    },
                    "description": "List of table names."
                }
            },
            "required": ["table_names"]
        }
    }
}

]


SYSTEM_PROMPT = """
You are a fantasy football statistical analyst.

You have access to two tools:
- get_schemas: retrieve column names and types for any allowed table in the database.
- run_query: run a SELECT query against the database.

Allowed tables: player_stats, snap_counts.

Rules:
- Always call get_schemas first to understand the table structure before querying.
- Never assume column names or data without querying.
- Use SQL SELECT statements only.
- Base all conclusions strictly on returned data.
- If insufficient data exists, say so.
- Always call a tool to retrieve data. Never describe a query without executing it.

## Column disambiguation
- player_stats has both `player_name` and `player_display_name`.
  - `player_name` is often an abbreviated form (e.g. "S.Barkley").
  - `player_display_name` is the full name (e.g. "Saquon Barkley").
  - ALWAYS filter on `player_display_name` when matching a full name from
    the user's question, unless the schema/data suggests otherwise.

## Multi-table comparisons
When a question asks to compare or combine data from two tables (e.g. snap
counts and performance stats), use a single SQL JOIN rather than separate
queries. Note the column names differ between tables:
- player_stats uses: player_display_name, season, week, season_type
- snap_counts uses:  player, season, week, game_type

Join example:
SELECT ps.week, ps.carries, ps.rushing_yards, ps.receiving_yards,
       ps.fantasy_points_ppr, sc.offense_snaps, sc.offense_pct
FROM player_stats ps
JOIN snap_counts sc
    ON ps.player_display_name = sc.player
    AND ps.season = sc.season
    AND ps.week = sc.week
WHERE ps.player_display_name = '<player>'
    AND ps.season = <year>
    AND ps.season_type = 'REG';

Your goal is to compare players and recommend optimal fantasy decisions.
"""


In [31]:
#@title Create Stats Agent

def ask_stats_agent(user_question):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question}
    ]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=stat_tools,
        tool_choice="auto"
    )

    message = response.choices[0].message
    print(message.tool_calls)
    # Agentic loop: keep handling tool calls until the model returns a final answer
    max_iterations = 5
    iteration = 0
    while message.tool_calls and iteration < max_iterations:
        iteration += 1
        messages.append(message)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if function_name == "run_query":
              try:
                print(f"Attempting SQL: {arguments['sql']}")
                result_df = run_query(arguments["sql"])
                print(result_df.head())
                tool_response = result_df.to_markdown(index=False)

              except Exception as e:
                tool_response = f"Query failed with exception {str(e)}. Please fix the SQL and try again."""

            elif function_name == "get_schemas":
                tool_response = get_schemas(arguments["table_names"])

            else:
                tool_response = f"Unknown tool: {function_name}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_response
            })

        # Let LLM interpret results and decide if more tool calls are needed

        next_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=stat_tools,
            tool_choice="auto"


)
        message = next_response.choices[0].message


    return message.content


# Stress Testing Stats Agent

**Player name resolution**
1. How many yards did Saquon Barkley have in 2024?
2. Stats for CMC in 2024
3. Stats for Michael Pittman in 2024
4. Stats for Saquan Barkley in 2024

**Aggregation correctness**

5. Total rushing yards for Derrick Henry across 2022-2024
6. Playoff rushing yards for Derrick Henry in 2023
7. What was Tyreek Hill's average yards per game in 2024?

**Position/grouping edge cases**

8. Top 5 fullbacks by receiving yards in 2024
9. Highest rushing total for an Eagles RB in 2024
10. Stats for Saquon Barkley in 2019

**Schema/column ambiguity**

11. Who had more targets than receptions by the widest margin in 2024?
12. Review snap counts and stats for Bijan Robinson in 2024


In [32]:
#ask_stats_agent("Review snap counts and stats for Bijan Robinson in 2024")


[ChatCompletionMessageFunctionToolCall(id='call_D2jrlWKp6G3N6fcNUIq8iPwN', function=Function(arguments='{"table_names": ["player_stats"]}', name='get_schemas'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_r2e0wPjXP2Ud84r8q7wOt8ip', function=Function(arguments='{"table_names": ["snap_counts"]}', name='get_schemas'), type='function')]
Attempting SQL: SELECT week, fantasy_points_ppr FROM player_stats WHERE player_display_name = 'Bijan Robinson' AND season = 2024 AND season_type = 'REG' ORDER BY week;
   week  fantasy_points_ppr
0     1                16.1
1     2                16.2
2     3                13.2
3     4                11.4
4     5                10.7


"Bijan Robinson's weekly fantasy points (PPR) for the 2024 regular season are as follows:\n\n- Week 1: 16.1\n- Week 2: 16.2\n- Week 3: 13.2\n- Week 4: 11.4\n- Week 5: 10.7\n- Week 6: 25.5\n- Week 7: 23.3\n- Week 8: 23.6\n- Week 9: 21.5\n- Week 10: 29.4\n- Week 11: 10.3\n- Week 13: 25.5\n- Week 14: 20.1\n- Week 15: 14.5\n- Week 16: 24.3\n- Week 17: 24.8\n- Week 18: 31.3\n\nNote: There is no data available for Week 12, which could indicate a bye week or an absence."

# Part 2 — News Agent

## Design

The stats agent answers: *"What injuries, lineup decisions, and/or off-field considerations are affecting this player?"* The LLM is prompted to consider the player in question, surrounding players, and opposing players.

It uses RAG via chromadb populated from ESPN news articles

The agent gets the tool:
- `retrieve_relevant_news` — Find the most relevant news articles to the player in question.

In [ ]:
# @title News — build_news_documents + refresh_news

# ---------------------------------------------------------------------------
# Staleness config
# ---------------------------------------------------------------------------
NEWS_TTL_SECONDS = 3 * 3600   # treat cache as stale after 3 hours

# Module-level cache: stores the timestamp of the last successful refresh
# so we can skip unnecessary re-fetches within the same session.
_news_last_refreshed_at: float | None = None   # epoch seconds (UTC)


def _news_cache_is_fresh() -> bool:
    """Return True if the in-memory ChromaDB collection was refreshed
    recently enough that we don't need to hit ESPN again."""
    if _news_last_refreshed_at is None:
        return False
    age = time.time() - _news_last_refreshed_at
    return age < NEWS_TTL_SECONDS


def _collection_has_docs() -> bool:
    """Return True if ChromaDB already has documents (persisted from a
    previous session), so we can skip a full re-fetch on cold start."""
    try:
        return news_collection.count() > 0
    except Exception:
        return False


def build_news_documents(espn_articles: list[dict]) -> list[dict]:
    """
    Convert raw ESPN articles into ChromaDB-ready documents.
    Each document contains headline, description, tagged players/teams,
    and a stable ID derived from the article URL.
    """
    documents = []

    for article in espn_articles:
        if not article["headline"] and not article["description"]:
            continue

        text_parts = [f"Headline: {article['headline']}"]

        if article["published"]:
            text_parts.append(f"Published: {article['published']}")
        if article["teams_mentioned"]:
            text_parts.append(f"Teams: {', '.join(article['teams_mentioned'])}")
        if article["players_mentioned"]:
            text_parts.append(f"Players mentioned: {', '.join(article['players_mentioned'])}")
        if article["description"]:
            text_parts.append(f"Summary: {article['description']}")
        if article["url"]:
            text_parts.append(f"Source: {article['url']}")

        text   = "\n".join(text_parts)
        doc_id = hashlib.md5(
            (article["url"] or article["headline"]).encode()
        ).hexdigest()

        documents.append({
            "id":   doc_id,
            "text": text,
            "metadata": {
                "headline":   article["headline"][:200],
                "published":  article["published"],
                "players":    ", ".join(article["players_mentioned"])[:200],
                "teams":      ", ".join(article["teams_mentioned"])[:100],
                "url":        article["url"][:500],
                "source":     "espn",
                "fetched_at": datetime.now(timezone.utc).isoformat(),
            }
        })

    print(f"[News] Built {len(documents)} documents from {len(espn_articles)} ESPN articles.")
    return documents


def refresh_news(espn_limit: int = 100, force: bool = False) -> None:
    """
    Refresh the news pipeline — fetch ESPN articles, embed them, and upsert
    into ChromaDB.

    Refresh is skipped when:
      - force=False  AND
      - the in-memory cache is still within NEWS_TTL_SECONDS  AND
      - ChromaDB already contains documents (so cold-start still fetches)

    Args:
        espn_limit: max ESPN articles to fetch (default 100)
        force:      bypass staleness check and always refresh
    """
    global _news_last_refreshed_at

    # --- Staleness gate -------------------------------------------------
    if not force and _news_cache_is_fresh():
        age_min = int((time.time() - _news_last_refreshed_at) / 60)
        print(f"[News] Cache is fresh ({age_min}m old, TTL={NEWS_TTL_SECONDS//3600}h). Skipping refresh.")
        print(f"[News] Documents in collection: {news_collection.count()}")
        return

    # On cold start (new session, persisted Chroma data from last run):
    # if docs already exist and this is the first call, give a light warning
    # but proceed — stale persisted data is worse than a fresh fetch.
    if _news_last_refreshed_at is None and _collection_has_docs():
        print(f"[News] Cold start: {news_collection.count()} docs found in ChromaDB. Refreshing to ensure freshness...")

    print("=" * 50)
    print("Refreshing news pipeline...")
    print("=" * 50)

    espn_articles = fetch_espn_news(limit=espn_limit)
    documents     = build_news_documents(espn_articles)

    batch_size = 100
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]

        # Deduplicate within the batch — ChromaDB rejects duplicate IDs
        # even in a single upsert call (it doesn't collapse them itself)
        seen = {}
        for d in batch:
            seen[d["id"]] = d   # last writer wins on true duplicates
        batch = list(seen.values())

        texts      = [d["text"]     for d in batch]
        ids        = [d["id"]       for d in batch]
        metas      = [d["metadata"] for d in batch]
        embeddings = embed_texts(texts)

        news_collection.upsert(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metas
        )
        print(f"  Upserted batch {i // batch_size + 1} ({len(batch)} docs)")

    _news_last_refreshed_at = time.time()
    print(f"\nDone. Total documents in collection: {news_collection.count()}")

# NOTE: Do NOT call refresh_news() here — call session_startup() instead.
print("News pipeline functions defined.")

In [ ]:
#@title Define News Agent and Tools

def retrieve_relevant_news(query: str, n_results: int = 5) -> str:
    """
    Retrieve relevant NFL news from ChromaDB.
    Includes source (ESPN) and publication date so the LLM can weight recency.
    """
    query_embedding = embed_texts([query])[0]

    results = news_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    if not results["documents"][0]:
        return "No relevant news found."

    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        relevance = round((1 - dist) * 100, 1)
        source    = meta.get("source", "unknown")
        published = meta.get("published", "")[:10]  # date only
        output.append(
            f"[Relevance: {relevance}% | Source: {source} | Date: {published}]\n{doc}"
        )

    return "\n\n---\n\n".join(output)


news_tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_relevant_news",
            "description": (
                "Search the NFL news vector store for injury reports and player updates. "
                "Pass a natural language query such as a player name or topic. "
                "You may call this multiple times with different queries to improve coverage."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Natural language search query, e.g. player name or injury topic."
                    },
                    "n_results": {
                        "type": "integer",
                        "description": "Number of results to return (default 5).",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

NEWS_SYSTEM_PROMPT = """
You are an NFL injury and news analyst for fantasy football. Your job is to
report all news relevant to the player(s) given the context of the question.

For example, if the question was "Is Saquon Barkley going to play this week?"
Then return information about his availability.
But if asked "Should I start Christian McCaffrey or Jahmyr Gibbs?"
You would want to look at not only their news/injuries, but also at their
quarterbacks, opposing defenses, etc.
Anything that might have an affect on their fantasy score.

You have access to a vector store of current NFL player news and injury reports
via the retrieve_relevant_news tool. All documents come from ESPN news articles.

Rules:
- Always call retrieve_relevant_news before answering any question about a player.
- You may call it multiple times with different queries if the first returns weak results
  (e.g. search by player name, then by team name, then by injury type).
- Always note the publication date of your source — older articles may be stale.
- Report injury status clearly: Active, Questionable, Doubtful, Out, IR.
- If no news is found, say so explicitly.
- Be concise — focus on what matters for fantasy decisions.
"""

def ask_news_agent(user_question: str) -> str:
    messages = [
        {"role": "system", "content": NEWS_SYSTEM_PROMPT},
        {"role": "user",   "content": user_question}
    ]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=news_tools,
        tool_choice="required"
    )
    message = response.choices[0].message

    max_iterations = 6
    iteration = 0

    while message.tool_calls and iteration < max_iterations:
        iteration += 1
        messages.append(message)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments     = json.loads(tool_call.function.arguments)

            if function_name == "retrieve_relevant_news":
                tool_response = retrieve_relevant_news(
                    query=arguments["query"],
                    n_results=arguments.get("n_results", 5)
                )
            else:
                tool_response = f"Unknown tool: {function_name}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_response
            })

        next_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=news_tools,
            tool_choice="auto"
        )
        message = next_response.choices[0].message

    return message.content

In [ ]:
ask_news_agent("Tell me about some notable injuries")

# Part 3 — Matchup Agent

## Design

The matchup agent answers: *"How good/bad is this player's opponent at stopping their position?"*

It works entirely from the `play_by_play` table.
The agent gets two tools:
- `get_matchup_schemas` — similar to `get_schemas` but substitutes column guidance for pbp as it has ~400 columns.
- `run_matchup_query` — same read-only guardrail as `run_query`, but adds `play_by_play` to allowed tables

In [ ]:
# @title Matchup agent — allowed tables and schema tool

MATCHUP_ALLOWED_TABLES = {
    "play_by_play",
    "player_stats",
    "players",
    "snap_counts",
}

# Key pbp columns the agent needs to know about, grouped by use case.
# This is injected into the system prompt — it's the view-layer substitute
# until proper SQL views are built (Part 3 of the roadmap).

PBP_COLUMN_GUIDE = """
## play_by_play — key columns by use case

### Identifiers / filters
- season          INT     — e.g. 2024
- week            INT     — 1-18 regular season, 19-22 playoffs
- game_id         VARCHAR — unique per game
- play_id         BIGINT  — unique per play
- posteam         VARCHAR — team with possession (offense)
- defteam         VARCHAR — defending team
- play_type       VARCHAR — 'pass', 'run', 'punt', 'field_goal', 'kickoff', 'no_play', etc.
- season_type     VARCHAR — 'REG' or 'POST'

### Passing / receiver analysis  (filter: play_type = 'pass')
- passer_player_name   VARCHAR  — QB who threw
- receiver_player_name VARCHAR  — target receiver
- pass_attempt         INT      — 1 if pass was attempted
- complete_pass        INT      — 1 if caught
- passing_yards        DOUBLE   — yards gained on the pass play (0 if incomplete)
- touchdown            INT      — 1 if TD scored
- interception         INT      — 1 if INT thrown
- air_yards            DOUBLE   — depth of target behind/past LOS
- yards_after_catch    DOUBLE   — YAC

### Rushing analysis  (filter: play_type = 'run')
- rusher_player_name   VARCHAR  — ball carrier
- rush_attempt         INT      — 1 if rush
- rushing_yards        DOUBLE   — yards gained
- touchdown            INT      — 1 if TD

### Opponent defensive ratings — CORRECT query patterns
Always filter season_type = 'REG' unless postseason is specifically requested.

Pass defense vs WR/TE (yards allowed per game to receivers):
  SELECT defteam, season, week, SUM(passing_yards) as pass_yds
  FROM play_by_play
  WHERE play_type = 'pass' AND season = 2024 AND season_type = 'REG'
  GROUP BY defteam, season, week
  -- then aggregate by defteam for per-game average

Rush defense (yards allowed per game to RBs):
  SELECT defteam, season, week, SUM(rushing_yards) as rush_yds
  FROM play_by_play
  WHERE play_type = 'run' AND season = 2024 AND season_type = 'REG'
  GROUP BY defteam, season, week

Points allowed by team:
  Use player_stats table instead — pbp TD counting is complex.

### Important gotchas
- DO NOT use yards_gained as a passing or rushing total — it mixes play types.
- DO NOT count touchdowns without filtering play_type — special teams TDs will pollute the count.
- Penalty plays have play_type = 'no_play' — exclude unless specifically studying penalties.
- Two-point conversions have play_type = 'pass' or 'run' but touchdown = 0, pts_earned_offense = 2.
"""


def get_matchup_schemas(table_names: list[str]) -> str:
    """Like get_schemas but allows play_by_play and injects the column guide."""
    invalid = [t for t in table_names if t not in MATCHUP_ALLOWED_TABLES]
    if invalid:
        raise ValueError(f"Tables not allowed: {invalid}")

    con = duckdb.connect(DB_PATH)
    try:
        output = []
        for table in table_names:
            df = con.execute(f"DESCRIBE {table}").df()
            # For play_by_play, return only the column guide rather than all 400 columns.
            # Full schema via DESCRIBE would exhaust the context window.
            if table == "play_by_play":
                output.append(
                    f"### play_by_play (curated column guide)\n{PBP_COLUMN_GUIDE}"
                )
            else:
                schema_md = df.to_markdown(index=False)
                output.append(f"### {table}\n{schema_md}\n")
    finally:
        con.close()

    return "\n".join(output)


def run_matchup_query(sql: str):
    """Read-only query runner; same guardrail as run_query."""
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE"]
    if any(word in sql.upper() for word in forbidden):
        raise ValueError("Dangerous SQL detected.")
    con = duckdb.connect(DB_PATH)
    try:
        result = con.execute(sql).df()
    finally:
        con.close()
    return result

In [ ]:
# @title Matchup agent — tool definitions and system prompt

matchup_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_matchup_schemas",
            "description": (
                "Retrieve schema and column guidance for matchup-relevant tables. "
                "For play_by_play, returns a curated column guide instead of the full 400-column schema. "
                "Allowed tables: play_by_play, player_stats, snap_counts, players."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "table_names": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "enum": ["play_by_play", "player_stats", "snap_counts", "players"]
                        },
                        "description": "Tables to retrieve schemas for."
                    }
                },
                "required": ["table_names"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_matchup_query",
            "description": "Run a read-only SQL query against matchup-relevant tables.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "A SELECT query."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]


MATCHUP_SYSTEM_PROMPT = """
You are a fantasy football matchup analyst. Your job is to evaluate how favorable or
unfavorable a player's upcoming opponent is for their position.

You have access to:
- get_matchup_schemas: retrieve column information for any allowed table
- run_matchup_query: run SELECT queries against the nflverse database

## Analysis approach by position

QB / WR / TE — evaluate the opponent's PASS defense:
  - Avg passing yards allowed per game (last 4 weeks and season-to-date)
  - Passing TDs allowed per game
  - Use play_by_play with play_type = 'pass'

RB — evaluate the opponent's RUSH defense:
  - Avg rushing yards allowed per game (last 4 weeks and season-to-date)
  - Rushing TDs allowed per game
  - Use play_by_play with play_type = 'run'
  - Also check if the opponent allows significant receiving work to RBs (pass plays where
    receiver position is RB — join via players table if needed)

## Rules
- Always call get_matchup_schemas for play_by_play before writing queries.
- Always filter season_type = 'REG' unless postseason is explicitly requested.
- Compute BOTH recent (last 4 weeks) and full-season averages — they often diverge.
- Express results as: yards/game, TDs/game, and a qualitative grade (Elite / Good / Average / Weak / Exploitable).
- If the current season has fewer than 4 completed weeks, use all available weeks.
- Never use yards_gained as a position-specific metric — always use passing_yards or rushing_yards.
- Always include row counts in your SQL (COUNT(DISTINCT game_id)) so the averages are verifiable.
- Return data in a format the synthesizer can use: team name, grade, key stats, and a 1-sentence summary.
"""

In [ ]:
# @title Matchup agent — ask_matchup_agent()

def ask_matchup_agent(user_question: str) -> str:
    """
    Evaluate a matchup from a fantasy perspective.

    Example questions:
      'How good is the Dallas defense against WR1s this season?'
      'Is the Chiefs matchup favorable for a RB this week?'
      'How does Jahmyr Gibbs do against the Packers defense historically?'
    """
    messages = [
        {"role": "system", "content": MATCHUP_SYSTEM_PROMPT},
        {"role": "user",   "content": user_question}
    ]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=matchup_tools,
        tool_choice="auto"
    )
    message = response.choices[0].message

    max_iterations = 8  # matchup queries often need 2-3 queries (season + recent + join)
    iteration = 0

    while message.tool_calls and iteration < max_iterations:
        iteration += 1
        messages.append(message)

        for tool_call in message.tool_calls:
            fn   = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if fn == "get_matchup_schemas":
                tool_response = get_matchup_schemas(args["table_names"])

            elif fn == "run_matchup_query":
                try:
                    print(f"[Matchup SQL] {args['sql'][:120]}...")
                    result_df = run_matchup_query(args["sql"])
                    # Pass full result for matchup aggregates (they're small)
                    # but cap at 32 rows (one per team)
                    tool_response = result_df.head(32).to_markdown(index=False)
                    tool_response += f"\n\n(Returned {len(result_df)} rows)"
                except Exception as e:
                    tool_response = f"Query failed: {str(e)}. Fix the SQL and retry."

            else:
                tool_response = f"Unknown tool: {fn}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_response
            })

        next_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=matchup_tools,
            tool_choice="auto"
        )
        message = next_response.choices[0].message

    return message.content


# Quick smoke test
# ask_matchup_agent("How good is the Dallas Cowboys pass defense this season? Grade it for fantasy.")

# Part 4 — Vegas Agent

## Design

The Vegas agent answers: *"What does the market say about scoring volume and individual player output for this game?"*

It has two data layers:
1. **Game lines** — implied team totals derived from spreads + game totals (fetched once per session, TTL-gated)
2. **Player props** — per-player market lines fetched on demand per game

The agent is tool-equipped so the LLM can resolve team names, fetch props for the relevant game,
and interpret both layers itself — rather than relying on fragile pre-processing in Python.


In [ ]:
# @title Vegas agent — data fetching (game lines + player props)

ODDS_API_KEY        = userdata.get("ODDS_API_KEY")
ODDS_BASE_URL       = "https://api.the-odds-api.com/v4"
PREFERRED_BOOKMAKER = "draftkings"

# Player prop market keys — each costs ~4 credits per game on the free tier
PROP_MARKETS = {
    "QB":  ["player_pass_yds", "player_pass_tds", "player_pass_attempts", "player_rush_yds"],
    "RB":  ["player_rush_yds", "player_rush_attempts", "player_reception_yds", "player_receptions"],
    "WR":  ["player_reception_yds", "player_receptions"],
    "TE":  ["player_reception_yds", "player_receptions"],
}

MARKET_LABELS = {
    "player_pass_yds":       "Passing yards",
    "player_pass_tds":       "Passing TDs",
    "player_pass_attempts":  "Pass attempts",
    "player_rush_yds":       "Rushing yards",
    "player_rush_attempts":  "Rush attempts",
    "player_reception_yds":  "Receiving yards",
    "player_receptions":     "Receptions",
}

# ---------------------------------------------------------------------------
# Staleness config — mirrors news TTL pattern
# ---------------------------------------------------------------------------
ODDS_TTL_SECONDS = 6 * 3600   # treat game-lines cache as stale after 6 hours
_cached_implied_totals: list[dict] | None = None
_odds_last_fetched_at:  float | None      = None


def _odds_cache_is_fresh() -> bool:
    if _odds_last_fetched_at is None:
        return False
    return (time.time() - _odds_last_fetched_at) < ODDS_TTL_SECONDS


# ── Game lines ────────────────────────────────────────────────────────────────

def fetch_nfl_odds() -> list[dict]:
    """Fetch current NFL game odds (totals + spreads). Costs 2 API credits."""
    url = f"{ODDS_BASE_URL}/sports/americanfootball_nfl/odds/"
    params = {
        "apiKey":     ODDS_API_KEY,
        "regions":    "us",
        "markets":    "totals,spreads",
        "oddsFormat": "american",
        "bookmakers": PREFERRED_BOOKMAKER,
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    remaining = r.headers.get("x-requests-remaining", "?")
    print(f"[Vegas] Game lines fetched. API credits remaining: {remaining}")
    return r.json()


def compute_implied_totals(games: list[dict]) -> list[dict]:
    """
    Derive per-team implied totals from raw game odds.

    Math:
      home_implied = (game_total - home_spread) / 2
      away_implied = game_total - home_implied
    """
    results = []
    for game in games:
        game_total  = None
        home_spread = None

        for bookmaker in game.get("bookmakers", []):
            for market in bookmaker.get("markets", []):
                if market["key"] == "totals":
                    game_total = market["outcomes"][0]["point"]
                elif market["key"] == "spreads":
                    for outcome in market["outcomes"]:
                        if outcome["name"] == game["home_team"]:
                            home_spread = outcome["point"]

        if game_total is None or home_spread is None:
            continue

        home_implied = round((game_total - home_spread) / 2, 1)
        away_implied = round(game_total - home_implied, 1)

        results.append({
            "game_id":       game["id"],
            "home_team":     game["home_team"],
            "away_team":     game["away_team"],
            "game_total":    game_total,
            "home_spread":   home_spread,
            "home_implied":  home_implied,
            "away_implied":  away_implied,
            "commence_time": game.get("commence_time", ""),
        })

    return sorted(results, key=lambda x: x["home_implied"] + x["away_implied"], reverse=True)


def get_implied_totals_cached(force: bool = False) -> list[dict]:
    """
    Return cached implied totals, refreshing if the cache is stale or missing.

    Odds are TTL-gated (ODDS_TTL_SECONDS). Pass force=True to bypass the gate.
    Returns an empty list (with a warning) if the API key is missing.
    """
    global _cached_implied_totals, _odds_last_fetched_at

    if not ODDS_API_KEY:
        print("[Vegas] WARNING: ODDS_API_KEY not set — no game lines available.")
        return []

    if not force and _odds_cache_is_fresh() and _cached_implied_totals is not None:
        age_min = int((time.time() - _odds_last_fetched_at) / 60)
        print(f"[Vegas] Odds cache is fresh ({age_min}m old). Skipping re-fetch.")
        return _cached_implied_totals

    raw = fetch_nfl_odds()
    _cached_implied_totals = compute_implied_totals(raw)
    _odds_last_fetched_at  = time.time()
    return _cached_implied_totals


# ── Player props ──────────────────────────────────────────────────────────────

def fetch_player_props(game_id: str, position: str) -> list[dict]:
    """
    Fetch player prop lines for a specific game and position group.

    Credit cost: ~4 credits per market per game. All position markets are
    batched into one call to minimise usage.
    """
    markets     = PROP_MARKETS.get(position.upper(), PROP_MARKETS["WR"])
    markets_str = ",".join(markets)

    url    = f"{ODDS_BASE_URL}/sports/americanfootball_nfl/events/{game_id}/odds"
    params = {
        "apiKey":     ODDS_API_KEY,
        "regions":    "us",
        "markets":    markets_str,
        "oddsFormat": "american",
        "bookmakers": PREFERRED_BOOKMAKER,
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"[Props] Fetch failed for game {game_id}: {e}")
        return []

    remaining = r.headers.get("x-requests-remaining", "?")
    print(f"[Props] Fetched {position} props for game {game_id}. Credits remaining: {remaining}")

    data  = r.json()
    props = []

    for bookmaker in data.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            market_key = market["key"]
            for outcome in market.get("outcomes", []):
                if outcome.get("description") not in ("Over", "Under"):
                    continue
                existing = next(
                    (p for p in props
                     if p["player"] == outcome["name"] and p["market"] == market_key),
                    None
                )
                if existing is None:
                    existing = {
                        "player":     outcome["name"],
                        "market":     market_key,
                        "line":       outcome.get("point"),
                        "over_odds":  None,
                        "under_odds": None,
                    }
                    props.append(existing)
                if outcome["description"] == "Over":
                    existing["over_odds"] = outcome["price"]
                else:
                    existing["under_odds"] = outcome["price"]

    return props


def format_game_lines_table(implied_totals: list[dict]) -> str:
    """Render implied totals as a plain-text table for LLM injection."""
    if not implied_totals:
        return "No NFL game lines available."

    header = (
        f"{'Game ID':<36} {'Home Team':<25} {'Away Team':<25} "
        f"{'Total':>6} {'Spread':>7} {'Home Imp':>9} {'Away Imp':>9}"
    )
    sep   = "-" * len(header)
    rows  = [header, sep]
    for g in implied_totals:
        rows.append(
            f"{g['game_id']:<36} {g['home_team']:<25} {g['away_team']:<25} "
            f"{g['game_total']:>6.1f} {g['home_spread']:>7.1f} "
            f"{g['home_implied']:>9.1f} {g['away_implied']:>9.1f}"
        )
    return "\n".join(rows)


def format_props_for_llm(props: list[dict], player_name: str) -> str:
    """Filter and format prop lines for a specific player."""
    player_lower  = player_name.lower()
    player_props  = [
        p for p in props
        if player_lower in p["player"].lower()
        or any(part in p["player"].lower() for part in player_lower.split())
    ]
    if not player_props:
        return f"No props found for {player_name}."

    lines = [f"Player props for {player_props[0]['player']}:"]
    for prop in player_props:
        label = MARKET_LABELS.get(prop["market"], prop["market"])
        lines.append(
            f"  {label}: {prop['line']} | Over {prop['over_odds']} / Under {prop['under_odds']}"
        )
    return "\n".join(lines)

print("Vegas data-fetching functions defined.")

In [ ]:
# @title Vegas agent — tools, system prompt, and ask_vegas_agent()

# ---------------------------------------------------------------------------
# Tool implementations callable by the LLM
# ---------------------------------------------------------------------------

def vegas_get_game_lines() -> str:
    """
    Return this week\'s implied team totals table (uses cached data).
    Called by the LLM when it needs to see the full slate.
    """
    totals = get_implied_totals_cached()
    if not totals:
        return "No game lines available (offseason or API key missing)."
    return format_game_lines_table(totals)


def vegas_get_player_props(game_id: str, position: str, player_name: str) -> str:
    """
    Fetch and return prop lines for a player in a specific game.
    game_id must be the exact Odds API game_id from the game-lines table.
    position must be QB, RB, WR, or TE.
    """
    if not ODDS_API_KEY:
        return "ODDS_API_KEY not configured — props unavailable."
    props = fetch_player_props(game_id, position)
    return format_props_for_llm(props, player_name)


# ---------------------------------------------------------------------------
# Tool schema
# ---------------------------------------------------------------------------

vegas_tools = [
    {
        "type": "function",
        "function": {
            "name": "vegas_get_game_lines",
            "description": (
                "Retrieve this week\'s NFL game lines as a table of implied team totals, "
                "game totals, and spreads. Always call this first — it also exposes the "
                "game_id values needed to fetch player props. "
                "Includes: game_id, home_team, away_team, game_total, home_spread, "
                "home_implied, away_implied."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []},
        }
    },
    {
        "type": "function",
        "function": {
            "name": "vegas_get_player_props",
            "description": (
                "Fetch player prop lines (yards, TDs, receptions) for a specific player "
                "in a specific game. You MUST call vegas_get_game_lines first to obtain "
                "the correct game_id. Props cost API credits — only call when a specific "
                "player is being evaluated."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "game_id": {
                        "type": "string",
                        "description": "The Odds API game_id from the game-lines table."
                    },
                    "position": {
                        "type": "string",
                        "enum": ["QB", "RB", "WR", "TE"],
                        "description": "Player position — determines which prop markets are fetched."
                    },
                    "player_name": {
                        "type": "string",
                        "description": "Player\'s full or partial name used to filter results."
                    }
                },
                "required": ["game_id", "position", "player_name"]
            }
        }
    }
]

# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------

VEGAS_SYSTEM_PROMPT = """
You are a fantasy football Vegas/odds analyst.

You have two tools:
- vegas_get_game_lines  : returns this week\'s implied team totals table.
                          Always call this first — it contains the game_id
                          values required to look up player props.
- vegas_get_player_props: fetches player-specific prop lines for a given game.
                          Only call when evaluating a specific player.
                          Props cost API credits — don't fetch unnecessarily.

## Workflow
1. Always call vegas_get_game_lines first.
2. Identify the relevant team(s) from the game-lines table using full team names.
3. If the question is about a specific player and their position is known or
   inferable, call vegas_get_player_props with the correct game_id.
4. Synthesize both data layers in your response.

## Interpreting implied team totals
- Implied team total = market estimate of how many points that team scores
- Higher implied total → more offensive volume → more fantasy opportunity
- QB  : benefits directly from a high implied total
- WR/TE: benefit from high implied total + negative game script (trailing = more passing)
- RB  : benefit from high implied total + POSITIVE game script (leading = more rushing)
  Note: large spread favorites may abandon the run early — flag this for RBs

## Interpreting player props
- The prop line IS the market\'s projection for that player\'s output
- It\'s more specific than the team total — weight it heavily when available
- Juice (odds) matters: -140 Over means the market leans toward the Over
- Props and implied totals should corroborate each other — flag when they diverge

## Grading scale (implied totals)
- >= 28: Elite game environment
- 25-27: Good
- 22-24: Average
- 19-21: Tough
- <  19: Very tough

## Output format
- Always report: implied total, opponent implied, game total, home/away status
- If props available: lead with the prop line and juice, then contextualize with team total
- Include a position-specific interpretation
- Be concise — 4-6 bullet points per player/team
"""

# ---------------------------------------------------------------------------
# Agent entry point — uniform signature matching all other agents
# ---------------------------------------------------------------------------

def ask_vegas_agent(user_question: str) -> str:
    """
    Interpret Vegas lines and player props for a fantasy question.

    Signature is uniform with ask_stats_agent / ask_news_agent / ask_matchup_agent:
    accepts only a natural language question; all data fetching is driven by
    the LLM via tools.

    The orchestrator node calls this with state["question"] — no extra kwargs needed.
    """
    messages = [
        {"role": "system", "content": VEGAS_SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=vegas_tools,
        tool_choice="required",   # always start with vegas_get_game_lines
    )
    message = response.choices[0].message

    max_iterations = 6
    iteration      = 0

    while message.tool_calls and iteration < max_iterations:
        iteration += 1
        messages.append(message)

        for tool_call in message.tool_calls:
            fn   = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if fn == "vegas_get_game_lines":
                tool_response = vegas_get_game_lines()

            elif fn == "vegas_get_player_props":
                tool_response = vegas_get_player_props(
                    game_id     = args["game_id"],
                    position    = args["position"],
                    player_name = args["player_name"],
                )

            else:
                tool_response = f"Unknown tool: {fn}"

            messages.append({
                "role":         "tool",
                "tool_call_id": tool_call.id,
                "content":      tool_response,
            })

        next_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=vegas_tools,
            tool_choice="auto",
        )
        message = next_response.choices[0].message

    return message.content

print("Vegas agent defined.")

In [ ]:
# @title Session startup — run once per session to warm data sources

def session_startup(
    refresh_news_data: bool = True,
    prefetch_odds:     bool = True,
    force_refresh:     bool = False,
) -> None:
    """
    Warm all data sources for the session.

    Both news and odds caches are TTL-gated — calls within the TTL window
    are no-ops unless force_refresh=True.

    Args:
        refresh_news_data: Attempt a news refresh (gated by NEWS_TTL_SECONDS).
                           Set False to skip entirely (e.g. you just ran it).
        prefetch_odds:     Fetch and cache implied totals (gated by ODDS_TTL_SECONDS).
                           Set False to skip (saves 2 API credits if called recently).
        force_refresh:     Bypass TTL gates for both news and odds.
                           Use after mid-week injury report drops or line movements.
    """
    print("[Startup] Warming data sources...")
    print()

    if refresh_news_data:
        refresh_news(espn_limit=100, force=force_refresh)
    else:
        print("[Startup] Skipping news refresh (refresh_news_data=False).")

    print()

    if prefetch_odds:
        totals = get_implied_totals_cached(force=force_refresh)
        if totals:
            print(f"[Startup] Cached implied totals for {len(totals)} games.")
        else:
            print("[Startup] No game lines available (offseason or API key missing).")
    else:
        print("[Startup] Skipping odds prefetch (prefetch_odds=False).")

    print()
    print("[Startup] All systems ready.")
    print()
    print("Re-run with force_refresh=True after major injury news or line moves.")


#session_startup()

In [ ]:
print("Doc count:", news_collection.count())

peek = news_collection.peek(limit=3)
for i in range(len(peek["ids"])):
    print(f"\n--- Doc {i} ---")
    print("Metadata:", peek["metadatas"][i])
    print("Text preview:", peek["documents"][i][:300])